# Week 4 — Improved Model: Baseline vs Improved Comparison

**Baseline (Week 2):** TF-IDF cosine similarity → Logistic Regression  
**V1 (Week 3):** MiniLM sentence-embedding cosine similarity → Logistic Regression  
**V2 (Week 4, improved):** 7 hybrid features (semantic similarity, word-level similarity, key-point coverage, word overlap, length features) → scaled, tuned Logistic Regression

The logic lives in `src/features.py`, `src/compare_models.py` and `src/grade.py`; this notebook runs it.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'

## 1. Train all three models and compare them
Takes a few minutes on CPU (MiniLM encodes ~10k texts). Writes `results/model_comparison.md` and saves the models.

In [2]:
import compare_models
compare_models.main()

c:\Users\HP\AI-Exam-Answer-Grader\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train (4733, 8), Validation (338, 8), Test (506, 8)

[1/3] Baseline: TF-IDF cosine + Logistic Regression (Week 2)

[2/3] V1: all-MiniLM-L6-v2 cosine + Logistic Regression (Week 3)


Batches: 100%|██████████| 73/73 [01:12<00:00,  1.01it/s]



[3/3] V2: hybrid features + tuned Logistic Regression (Week 4)


Batches: 100%|██████████| 11/11 [00:06<00:00,  1.69it/s]


    C=0.1   class_weight=None      -> validation macro-F1 = 0.5735
    C=0.1   class_weight=balanced  -> validation macro-F1 = 0.5519
    C=1.0   class_weight=None      -> validation macro-F1 = 0.5766
    C=1.0   class_weight=balanced  -> validation macro-F1 = 0.5546
    C=10.0  class_weight=None      -> validation macro-F1 = 0.5766
    C=10.0  class_weight=balanced  -> validation macro-F1 = 0.5546
    Selected: C=1.0, class_weight=None

# Week 4 - Baseline vs Improved Model Comparison

Dataset: EngSAF (train 4733 / validation 338 / test 506). Labels: 0 = incorrect, 1 = partially correct, 2 = correct.

Metrics: Accuracy; Macro-F1 (treats the three classes equally); QWK = quadratic weighted kappa (agreement with the human grade, penalises big mistakes more); MAE = mean absolute error in label points (lower is better); Partial recall = share of 'partially correct' answers the model recognised.

## Test set

| Model | Accuracy | Macro-F1 | QWK | MAE | Partial recall |
|---|---|---|---|---

## 2. Comparison report

In [3]:
from IPython.display import Markdown, display
display(Markdown(open('../results/model_comparison.md', encoding='utf-8').read()))

# Week 4 - Baseline vs Improved Model Comparison

Dataset: EngSAF (train 4733 / validation 338 / test 506). Labels: 0 = incorrect, 1 = partially correct, 2 = correct.

Metrics: Accuracy; Macro-F1 (treats the three classes equally); QWK = quadratic weighted kappa (agreement with the human grade, penalises big mistakes more); MAE = mean absolute error in label points (lower is better); Partial recall = share of 'partially correct' answers the model recognised.

## Test set

| Model | Accuracy | Macro-F1 | QWK | MAE | Partial recall |
|---|---|---|---|---|---|
| Baseline - TF-IDF + LR (Week 2) | 0.5020 | 0.4688 | 0.3194 | 0.6225 | 0.2408 |
| V1 - MiniLM + LR (Week 3) | 0.5000 | 0.4752 | 0.3326 | 0.5988 | 0.3141 |
| V2 - Hybrid features + LR (Week 4) | 0.5257 | 0.5005 | 0.3643 | 0.5711 | 0.3770 |

## Validation set

| Model | Accuracy | Macro-F1 | QWK | MAE | Partial recall |
|---|---|---|---|---|---|
| Baseline - TF-IDF + LR (Week 2) | 0.5059 | 0.4820 | 0.4014 | 0.5888 | 0.1875 |
| V1 - MiniLM + LR (Week 3) | 0.5059 | 0.4966 | 0.4452 | 0.5533 | 0.2422 |
| V2 - Hybrid features + LR (Week 4) | 0.5828 | 0.5766 | 0.4793 | 0.4941 | 0.3750 |

## Confusion matrices (test set)

Rows = true label, columns = predicted label (0 / 1 / 2).

**Baseline - TF-IDF + LR (Week 2)**

| true \ pred | 0 | 1 | 2 |
|---|---|---|---|
| 0 | 48 | 18 | 47 |
| 1 | 26 | 46 | 119 |
| 2 | 16 | 26 | 160 |

**V1 - MiniLM + LR (Week 3)**

| true \ pred | 0 | 1 | 2 |
|---|---|---|---|
| 0 | 41 | 26 | 46 |
| 1 | 21 | 60 | 110 |
| 2 | 4 | 46 | 152 |

**V2 - Hybrid features + LR (Week 4)**

| true \ pred | 0 | 1 | 2 |
|---|---|---|---|
| 0 | 44 | 34 | 35 |
| 1 | 26 | 72 | 93 |
| 2 | 14 | 38 | 150 |

## Ablation (test set, same classifier settings as V2)

| Features used | Accuracy | Macro-F1 | QWK |
|---|---|---|---|
| MiniLM cosine only | 0.4941 | 0.4695 | 0.3329 |
| + TF-IDF cosine | 0.5000 | 0.4749 | 0.3286 |
| + word overlap + length features | 0.5316 | 0.5088 | 0.3793 |
| + key-point coverage (full V2) | 0.5257 | 0.5005 | 0.3643 |

## V2 feature weights for the 'correct' class

Features are standardised, so weights are comparable. Positive = pushes the prediction towards 'correct'.

| Feature | Weight |
|---|---|
| minilm_cosine | +0.511 |
| tfidf_cosine | -0.038 |
| keypoint_coverage | -0.058 |
| word_overlap | +0.439 |
| log_student_len | +0.292 |
| log_reference_len | -0.298 |
| length_ratio | -0.195 |

V2 hyper-parameters (chosen on validation macro-F1): C=1.0, class_weight=None


## 3. Grade a single answer with all three models

In [4]:
import joblib
from features import TextEncoder
from grade import V2_NLP_Pipeline

bundle = joblib.load('../models/grader_models.joblib')
pipeline = V2_NLP_Pipeline(bundle, TextEncoder(bundle['embedding_model']))

q_id = 'Q4 What does supervised learning use to train model?'
reference_ans = 'A supervised machine learning algorithm relies on labeled training data to learn a function that maps inputs to outputs.'
student_ans = 'Supervised learning uses labeled data to train models to predict outcomes.'

print('Baseline:', pipeline.grade_with_baseline(q_id, student_ans, reference_ans, 10))
print('V1:', pipeline.grade_with_v1(q_id, student_ans, reference_ans, 10))
print('V2:', pipeline.grade_answer(q_id, student_ans, reference_ans, 10))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2989.65it/s]


Baseline: {
    "question": "Q4 What does supervised learning use to train model?",
    "score": 5,
    "maximum_score": 10,
    "feedback": "The answer is partially correct but misses some key concepts.",
    "confidence": 0.39,
    "model": "baseline_tfidf"
}
V1: {
    "question": "Q4 What does supervised learning use to train model?",
    "score": 10,
    "maximum_score": 10,
    "feedback": "The answer covers all core concepts present in the reference.",
    "confidence": 0.6,
    "model": "v1_minilm"
}
V2: {
    "question": "Q4 What does supervised learning use to train model?",
    "score": 10,
    "maximum_score": 10,
    "feedback": "The answer covers all core concepts present in the reference.",
    "confidence": 0.6,
    "model": "v2_hybrid",
    "missing_key_points": []
}


## 4. A partially correct answer (shows missing key points)

In [5]:
reference_ans = ('- Write to CR3 register to modify page directory pointer\n'
                 '- Write to the IDTR register to change location of IDT\n'
                 '- Access to VA that lies in the kernel virtual address range')
student_ans = 'Writing to the CR3 register to change the page directory.'
print(pipeline.grade_answer('Q2 Privileged operations', student_ans, reference_ans, 6))

{
    "question": "Q2 Privileged operations",
    "score": 6,
    "maximum_score": 6,
    "feedback": "The answer is graded correct overall, but these reference points were not found: write to the idtr register to change location of idt; access to va that lies in the kernel virtual address range.",
    "confidence": 0.45,
    "model": "v2_hybrid",
    "missing_key_points": [
        "write to the idtr register to change location of idt",
        "access to va that lies in the kernel virtual address range"
    ]
}
